#### Visão Geral
##### Schema : gold
##### Table : run_job_case_gold

| Detalhe | Informação |
|---------|------------|
| Criado Originalmente Por | Wellikiandre Bosich |
| Tabela de Dados de Saída | `{environment}.gold.run_job_case_gold` |
| Origem Fonte de Dados de Entrada | Camada silver |
| Destino Fonte de Dados de Saída | Camada gold |

#### Histórico

| Data       | Desenvolvido Por         | Motivo                                         |
|:----------:|--------------------------|-----------------------------------------------|
| 04/06/2026 | Wellikiandre Bosich    | Orquestração geral e encadeamento das dimensões e fatos na Gold. |

In [ ]:
%run ../0_Config/0-Init

In [ ]:
def run_notebook(notebook_name: str) -> None:
    dbutils.notebook.run(notebook_name, 0)

In [ ]:
# Carga das Dimensões em Paralelo
from threading import Thread
from queue import Queue

dims = ["dim_clientes", "dim_produtos", "dim_vendedores", "dim_tempo"]
q = Queue()
for d in dims:
    q.put(d)

def worker():
    while not q.empty():
        job = q.get()
        print(f"▶ Dimensão: {job}")
        run_notebook(job)
        q.task_done()

for _ in range(4):
    t = Thread(target=worker)
    t.daemon = True
    t.start()

q.join()
print("Todas as dimensões carregadas. Iniciando fatos...")

In [ ]:
# Carga das Fatos sequencialmente
fatos = ["fact_pedidos_itens", "fact_entregas", "fact_ocorrencias"]
for f in fatos:
    print(f"▶ Fato: {f}")
    run_notebook(f)
print("Todas as tabelas da camada Gold foram atualizadas.")